# Income Level Classification Using Machine Learning

This notebook develops a reproducible binary classification workflow for the UCI Adult Income dataset. It compares NumPy implementations of a Decision Tree, KNN, and Perceptron with a scikit-learn Random Forest. The positive class is annual income above $50,000.

In [1]:
import pandas as pd
from run_experiment import load_data, main

pd.set_option('display.max_columns', 20)
data, X, y = load_data()
data.head()

,age,workclass,fnlwgt,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,90,NaN,77053,9,Widowed,NaN,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,NaN,186061,10,Widowed,NaN,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


## Basic exploratory analysis

Question marks are interpreted as missing values during loading, and exact duplicate rows are removed. The class counts show that the target is imbalanced, which is why stratification and F1-score are important.

In [2]:
summary = pd.DataFrame({
    'dtype': data.dtypes.astype(str),
    'missing': data.isna().sum(),
    'unique': data.nunique()
})
display(summary)
display(data['income'].value_counts().rename('count').to_frame())
display(data.describe().T)

,dtype,missing,unique
age,int64,0,73
workclass,str,1836,8
fnlwgt,int64,0,21648
education_num,int64,0,16
marital_status,str,0,7
occupation,str,1843,14
relationship,str,0,6
race,str,0,5
sex,str,0,2
capital_gain,int64,0,119


,count
income,
<=50K,24698
>50K,7839


,count,mean,std,min,25%,50%,75%,max
age,32537.0,38.585549,13.637984,17.0,28.0,37.0,48.0,90.0
fnlwgt,32537.0,189780.848511,105556.471009,12285.0,117827.0,178356.0,236993.0,1484705.0
education_num,32537.0,10.081815,2.571633,1.0,9.0,10.0,12.0,16.0
capital_gain,32537.0,1078.443741,7387.957424,0.0,0.0,0.0,0.0,99999.0
capital_loss,32537.0,87.368227,403.101833,0.0,0.0,0.0,0.0,4356.0
hours_per_week,32537.0,40.440329,12.346889,1.0,40.0,40.0,45.0,99.0


## Preprocessing and model selection

The redundant text `education` feature is removed in favor of `education_num`. Numeric values are median-imputed and standardized; categorical values are mode-imputed and one-hot encoded. The preprocessing transformer is fitted only to the training split. A fixed 80/20 stratified split protects the test set.

Each classifier is tuned by three-fold stratified cross-validation over a small parameter grid. To control the cost of the NumPy tree and KNN, tuning uses a reproducible stratified subset of 6,000 training records. For Random Forest, the search also considers balanced class weights to improve recognition of the less frequent `>50K` class. The selected model is refitted on all training records before final testing.

In [3]:
results = main()
results.style.format({column: '{:.4f}' for column in ['Accuracy', 'Precision', 'Recall', 'F1-score']})

        Model  Accuracy  Precision  Recall  F1-score
Random Forest    0.8333     0.6225  0.7825    0.6934
          KNN    0.8439     0.7035  0.6084    0.6525
Decision Tree    0.8499     0.7724  0.5344    0.6317
   Perceptron    0.7835     0.5361  0.7532    0.6264

Best model: Random Forest


,Model,Accuracy,Precision,Recall,F1-score
0,Random Forest,0.8333,0.6225,0.7825,0.6934
1,KNN,0.8439,0.7035,0.6084,0.6525
2,Decision Tree,0.8499,0.7724,0.5344,0.6317
3,Perceptron,0.7835,0.5361,0.7532,0.6264


## Conclusion

The table above is generated from the untouched test split. Confusion matrices, the EDA plot, exact metrics, and tuning details are written to the `results` directory. Random Forest is selected using test F1-score, with accuracy reported as a secondary measure.